# Evaluate a Function on a Grid

In [ ]:
import minterpy as mp
import numpy as np

Calling an instance of `Grid` with a function or a `Callable` evaluates the given function on the unisolvent nodes transformed to the grid's domain and returns the corresponding function values. In the context of polynomial interpolation, these function values are the coefficients of a polynomial in the Lagrange basis. If no domain is specified, the default domain $[-1, 1]^m$ is used and no transformation takes place.

This guide demonstrates how to call an instance of `Grid` on a function.

## Example: Function with one-dimensional output

Consider the following problem:

Compute the formula

$$
f(\boldsymbol{x}) = \sum_{i = 1}^{3} x_i^2, \boldsymbol{x} \in [-1, 1]^3 
$$

on the interpolation grid that corresponds to a complete multi-index set of
polynomial degree $3$ with respect to the $l_p$-degree $2.0$. 

### Function to evaluate

A valid function or callable that can be called with a `Grid` instance
must satisfy the following:

- it must accept as the first argument a two-dimensional array where
  each row corresponds to different evaluation points and each column
  corresponds to different spatial dimension.
- it must return an array with the same length as the input.

The function or callable may accept additional arguments either positional
or keyword.

The function as required above can be defined as follows:

In [ ]:
def fun_one_dim(xx: np.ndarray) -> np.ndarray:
    """Compute the sum of squares."""
    return np.sum(xx**2, axis=1)

### Grid

The interpolation grid that corresponds to the complete multi-index set
can be created using `from_degree()` factory method:

In [ ]:
spatial_dimension = 3
poly_degree = 2
lp_degree = 2.0
grd = mp.Grid.from_degree(spatial_dimension, poly_degree, lp_degree)

### Function values on the grid

By calling the `Grid` instance on the function defined above, we evaluate the function on the unisolvent nodes transformed to the domain of the grid:

In [ ]:
fun_values = grd(fun_one_dim)

In [ ]:
fun_values

Because the domain of the grid is the default domain $[-1, 1]^m$, there is no transformation involved and the above values are the same as evaluating the function directly on the unisolvent nodes of the grid:

In [ ]:
fun_one_dim(grd.unisolvent_nodes)

## Example: Function with multi-dimensional output

Consider the following problem:

Compute the formula

$$
\boldsymbol{f}(\boldsymbol{x}) = (f_1(\boldsymbol{x}), f_2(\boldsymbol{x}) ), \; \boldsymbol{x} \in [-1, 1]^3 
$$

where

$$
\begin{aligned}
f_1(\boldsymbol{x}) & = \sum_{i = 1}^{3} x_i^2,\\
f_2(\boldsymbol{x}) & = \prod_{i = 1}^{3} x_i^2,
\end{aligned}
$$

on the same interpolation grid as before.

Notice that the function now returns two outputs per input value.

The function or callable passed to the `Grid` instance may also return 
multiple outputs. The function must be defined such that it returns an array 
whose each column corresponds to the different output.


The required function can therefore be defined as follows:

In [ ]:
def fun_two_dim(xx: np.ndarray) -> np.ndarray:
    """Return the sum and product of squared."""
    yy = np.empty((len(xx), 2))
    
    yy[:, 0] = np.sum(xx**2, axis=1)
    yy[:, 1] = np.prod(xx**2, axis=1)
    
    return yy

The previous instance of `Grid` can be directly used to obtain the values
of the multiple-output function:

In [ ]:
grd(fun_two_dim)

As expected, calling the `Grid` instance with the function returns a two-dimensional
array whose each column corresponds to a different output and each row corresponds to a different unisolvent node.

## Example: Function with additional arguments

While the function passed to a `Grid` instance must take as its first
argument a two-dimensional array, additional arguments may also be passed
to the function by passing positional and keyword arguments to the call.

For instance, suppose the function to be evaluated is defined as follows:

In [ ]:
def fun_with_args(xx: np.ndarray, p: float) -> np.ndarray:
    """Return the row-wise lp-norm."""
    return np.sum(np.abs(xx**p), axis=1)**(1/p)

To change the behavior of the function call via one of its argument
when the function is evaluated on the grid, pass the additional arguments
to the call to the `Grid` instance.

For instance, with the additional argument to `fun_with_args()` as a positional argument:

In [ ]:
grd(fun_with_args, 1.0)

In [ ]:
grd(fun_with_args, 2.0)

...and as a keyword argument:


grd(fun_with_args, p=3.0)

## Example: Function on a custom rectangular domain

Consider now the same function defined on a custom rectangular domain:

$$
f(\boldsymbol{x}) = \sum_{i = 1}^3 x_i^2, \; \boldsymbol{x} = [0, 1] \times [1, 2] \times [-1, 1].
$$

The function definition remains unchanged but now a custom domain must be defined:

In [ ]:
dom = mp.Domain(
    np.array([
        [0, 1],
        [1, 2],
        [-1, 1],
    ])
)

and passed to the grid:

In [ ]:
grd_custom_domain = mp.Grid.from_degree(spatial_dimension, poly_degree, lp_degree, domain=dom)

By calling the `Grid` instance on the function defined above, we evaluate the function on the unisolvent nodes of the grid transformed to the domain of the function:

In [ ]:
grd_custom_domain(fun_one_dim)

These values are the same as:

In [ ]:
fun_one_dim(dom.map_from_internal(grd_custom_domain.unisolvent_nodes))